
Find the most repeated word from a text file

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("wordcount") \
    .getOrCreate()


In [5]:
df = spark.read.text("/workspaces/codespaces-jupyter/sample.tx")
df.show(truncate=False)


+-------------------------------------------------+
|value                                            |
+-------------------------------------------------+
|hello world hello spark hello pyspark world spark|
+-------------------------------------------------+



In [10]:
from pyspark.sql.functions import split, explode, col

# 3. split lines into words
words = df.select(explode(split(col("value"), " ")).alias("word"))

# 4. count words
word_counts = words.groupBy("word").count()

# 5. get most repeated word
word_counts.orderBy(col("count").desc()).show(2)


+-----+-----+
| word|count|
+-----+-----+
|hello|    3|
|spark|    2|
+-----+-----+
only showing top 2 rows


In [9]:

word_counts.count()


4

Create an RDD

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("RDD-demo").getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/23 17:24:38 WARN Utils: Your hostname, codespaces-b99eb4, resolves to a loopback address: 127.0.0.1; using 10.0.1.191 instead (on interface eth0)
25/11/23 17:24:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/23 17:24:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Now create a simple RDD

1.From a Python list

In [2]:
data = [("Somesh", 25), ("Rohan", 30), ("Anita", 28)]

rdd = spark.sparkContext.parallelize(data)


2.Create numbers RDD

In [5]:
rdd = spark.sparkContext.parallelize(range(1, 6))
rdd.collect()


[1, 2, 3, 4, 5]

Convert RDD → DataFrame

create a DataFrame from an existing RDD:

In [ ]:
df = rdd.toDF()
df.show()
df.printSchema()


The StructType and StructField classes in PySpark are used to define the schema for a DataFrame and create complex columns such as nested struct, array, and map columns. StructType is a collection of StructField objects that determine the column name, column data type, field nullability, and metadata.

PySpark imports the StructType class from pyspark.sql.types to describe the data frame’s structure. The data frames printSchema() function displays StructType columns as “struct.”

To define the columns, PySpark offers the StructField class from pyspark.sql.types, which includes the column name (String), column type (DataType)

Here is an example showing the use of StructType and StructField classes in PySpark:

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

data = [
    ("James", "", "William", "36636", "M", 3000),
    ("Michael", "Smith", "", "40288", "M", 4000),
    ("Robert", "", "Dawson", "42114", "M", 4000),
    ("Maria", "Jones", "", "39192", "F", 4000)
]

schema = StructType([
    StructField("firstname", StringType(), True),
    StructField("middlename", StringType(), True),
    StructField("lastname", StringType(), True),
    StructField("id", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("salary", IntegerType(), True)
])

df = spark.createDataFrame(data=data, schema=schema)
df.printSchema()
df.show(truncate=False)

root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)

+---------+----------+--------+-----+------+------+
|firstname|middlename|lastname|id   |gender|salary|
+---------+----------+--------+-----+------+------+
|James    |          |William |36636|M     |3000  |
|Michael  |Smith     |        |40288|M     |4000  |
|Robert   |          |Dawson  |42114|M     |4000  |
|Maria    |Jones     |        |39192|F     |4000  |
+---------+----------+--------+-----+------+------+



handle row duplication in a PySpark DataFrame?
There are two primary ways to handle row duplication in PySpark DataFrames. The distinct() function in PySpark is used to remove duplicate rows across all columns in a DataFrame, while dropDuplicates() is used to remove rows based on one or more specific columns.

In [13]:
# Distinct
distinctDF = df.distinct()
print(distinctDF.count())
distinctDF.show(truncate=False)

# Drop duplicates across all columns
df2 = df.dropDuplicates()
print(df2.count())
df2.show(truncate=False)


4
+---------+----------+--------+-----+------+------+
|firstname|middlename|lastname|id   |gender|salary|
+---------+----------+--------+-----+------+------+
|James    |          |William |36636|M     |3000  |
|Michael  |Smith     |        |40288|M     |4000  |
|Robert   |          |Dawson  |42114|M     |4000  |
|Maria    |Jones     |        |39192|F     |4000  |
+---------+----------+--------+-----+------+------+

4
+---------+----------+--------+-----+------+------+
|firstname|middlename|lastname|id   |gender|salary|
+---------+----------+--------+-----+------+------+
|James    |          |William |36636|M     |3000  |
|Michael  |Smith     |        |40288|M     |4000  |
|Robert   |          |Dawson  |42114|M     |4000  |
|Maria    |Jones     |        |39192|F     |4000  |
+---------+----------+--------+-----+------+------+



Cluster Mode:

Scenario: When the client computers are not located near the cluster.
Reason: This mode prevents network delays that would occur in Client mode due to communication between executors and the client machine. Additionally, in Cluster mode, if the client machine goes offline, the operation continues unaffected, as the driver runs within the cluster.

Client Mode:

Scenario: When the client computer is located within the cluster.
Reason: In this mode, there are no network latency issues because the client machine is part of the cluster. Maintenance of the cluster is already managed, so there’s no concern about operation loss if the client machine experiences a failure.

In summary, Cluster mode is preferred when the client is remote from the cluster to avoid network delays and ensure reliability. Client mode is suitable when the client is within the cluster, eliminating network latency concerns and simplifying maintenance.

MapReduce works only in batch mode and processes data slowly because it reads and writes everything to disk (HDFS). This makes it high-latency. Spark is much faster because it keeps most data in memory (RAM), supports caching, and can handle both batch and real-time (streaming) workloads. In simple terms: MapReduce = slow, disk-based batch processing; Spark = fast, memory-based batch + real-time processing.

You can load multiple files of the same type into one DataFrame?

If all your files are in a folder like:

/data/files/
   file1.csv
   file2.csv
   file3.csv

Use wildcard (*) to load specific pattern

In [ ]:
df = spark.read.csv("/data/files/*.csv", header=True, inferSchema=True)


What is a Cluster Manager?

A Cluster Manager is a component that controls and manages all the machines (nodes)

It decides:

Where to run your Spark jobs
How many resources (CPU, RAM) each job gets
Who gets priority if many jobs are running
Which worker node should execute which task

Types of Cluster Managers in Spark

1️⃣ Standalone — Spark’s own built-in manager(used in databricks)
2️⃣ YARN — Hadoop’s resource manager
3️⃣ Mesos — General cluster manager
4️⃣ Kubernetes — Most popular now for cloud deployments